In [3]:
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [10]:
import re  # regex to clean up newlines
import hashlib  # build a stable ID (fingerprint) from text
from pathlib import Path  # get file name from path

from langchain_community.document_loaders import PyPDFDirectoryLoader  # loads every PDF in a folder
from langchain_text_splitters import RecursiveCharacterTextSplitter  # splits text into small chunks
from langchain_ollama import OllamaEmbeddings, ChatOllama  # Ollama embedding and chat model
from langchain_chroma import Chroma  # vector database

# ---------- 1. Settings (one place, easy to change) ----------
CV_FOLDER = "../data/cvs"  # path to the CV folder
CHUNK_SIZE = 800  # max ~800 characters per chunk
CHUNK_OVERLAP = 100  # ~100 character overlap with the neighboring chunk
COLLECTION = "cv_chunks"  # vector store collection name
EMBED_MODEL = "nomic-embed-text"  # model used for embeddings
CHAT_MODEL = "llama3.2:3b"  # model used to answer questions

In [11]:
from langchain_ollama import OllamaEmbeddings
from langchain_ollama import OllamaLLM

# Configure Ollama LLM
ollama_llm = OllamaLLM(
    model="llama3.2:3b",
    base_url="http://localhost:11434",
    temperature=0.1
)

# Configure embedding model
ollama_embedding = OllamaEmbeddings(
    model="nomic-embed-text:latest",  # Correct parameter name is `model`
    base_url="http://localhost:11434",  # Base URL for the Ollama service
)

In [12]:
# ---------- 2. Load PDFs ----------
docs = PyPDFDirectoryLoader(CV_FOLDER, glob="**/*.pdf").load()  # one Document per page of each PDF

In [13]:

print(len(docs))
print(docs)

3
[Document(metadata={'producer': 'Skia/PDF m156 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Ashikujjaman Shaikat Resume', 'source': '../data/cvs/Ashikujjaman Shaikat Resume.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='Md\n \nAshikujjaman\n \nShaikat\n \n                           \n \n  \n PROFESSIONAL  SUMMARY  Senior  Software  Engineer  with  5+  years  of  professional  experience  building  scalable,  responsive,  and  maintainable  web  \napplications\n \nusing\n \nReact,\n \nVue,\n \nNext.js,\n \nJavaScript,\n \nand\n \nTypeScript.\n \nExperienced\n \nin\n \ndeveloping\n \nreusable\n \nUI\n \ncomponents,\n \nimplementing\n \nstate\n \nmanagement,\n \nand\n \ntranslating\n \nUI/UX\n \ndesigns\n \ninto\n \naccessible,\n \npixel-perfect\n \ninterfaces.\n \nStrong\n \nunderstanding\n \nof\n \nfrontend\n \narchitecture,\n \nperformance\n \noptimization,\n \nAPI\n \nintegration,\n \nand\n \ncross-functional\n \ncollaboration\n \nwi

In [14]:
for d in docs:  # for every page
    d.page_content = re.sub(r"\s+", " ", d.page_content).strip()  # collapse all newlines/extra spaces into one space
    d.metadata["candidate"] = Path(d.metadata["source"]).stem  # file name without extension


In [15]:
print(len(docs))
print(docs)

3
[Document(metadata={'producer': 'Skia/PDF m156 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Ashikujjaman Shaikat Resume', 'source': '../data/cvs/Ashikujjaman Shaikat Resume.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'candidate': 'Ashikujjaman Shaikat Resume'}, page_content='Md Ashikujjaman Shaikat PROFESSIONAL SUMMARY Senior Software Engineer with 5+ years of professional experience building scalable, responsive, and maintainable web applications using React, Vue, Next.js, JavaScript, and TypeScript. Experienced in developing reusable UI components, implementing state management, and translating UI/UX designs into accessible, pixel-perfect interfaces. Strong understanding of frontend architecture, performance optimization, API integration, and cross-functional collaboration with international teams. SKILLS SUMMARY ❖ Frontend Technology: ❖ UI & Styling: ❖ State Management: ❖ Testing & Code Quality: ❖ Architecture & Design: ❖ Backend & APIs: ❖ Dat

In [16]:
# ---------- 4. Split ----------
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,  # max chunk size
    chunk_overlap=CHUNK_OVERLAP,  # overlap
    is_separator_regex=True,  # the separators below are regex
    separators=[
        r"\s+(?=●)",  # split before ● (bullet stays with the text after it)
        r"(?<=[.!?])\s+",  # split at end of sentence
        r" ",  # split at word boundaries
        r"",  # last resort: split by character
    ],
)
chunks = splitter.split_documents(docs) 

In [17]:
print(len(chunks), "chunks created from", len(docs), "pages of CVs")
print(chunks)

11 chunks created from 3 pages of CVs
[Document(metadata={'producer': 'Skia/PDF m156 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Ashikujjaman Shaikat Resume', 'source': '../data/cvs/Ashikujjaman Shaikat Resume.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'candidate': 'Ashikujjaman Shaikat Resume'}, page_content='Md Ashikujjaman Shaikat PROFESSIONAL SUMMARY Senior Software Engineer with 5+ years of professional experience building scalable, responsive, and maintainable web applications using React, Vue, Next.js, JavaScript, and TypeScript. Experienced in developing reusable UI components, implementing state management, and translating UI/UX designs into accessible, pixel-perfect interfaces. Strong understanding of frontend architecture, performance optimization, API integration, and cross-functional collaboration with international teams.'), Document(metadata={'producer': 'Skia/PDF m156 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 

In [18]:
# ---------- 5. Attach candidate name to every chunk ----------
for c in chunks:
    c.page_content = f"CV of {c.metadata['candidate']}:\n{c.page_content}"  # so the LLM knows whose CV this is

In [19]:
print(len(chunks), "chunks created from", len(docs), "pages of CVs")
print(chunks)

11 chunks created from 3 pages of CVs
[Document(metadata={'producer': 'Skia/PDF m156 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Ashikujjaman Shaikat Resume', 'source': '../data/cvs/Ashikujjaman Shaikat Resume.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'candidate': 'Ashikujjaman Shaikat Resume'}, page_content='CV of Ashikujjaman Shaikat Resume:\nMd Ashikujjaman Shaikat PROFESSIONAL SUMMARY Senior Software Engineer with 5+ years of professional experience building scalable, responsive, and maintainable web applications using React, Vue, Next.js, JavaScript, and TypeScript. Experienced in developing reusable UI components, implementing state management, and translating UI/UX designs into accessible, pixel-perfect interfaces. Strong understanding of frontend architecture, performance optimization, API integration, and cross-functional collaboration with international teams.'), Document(metadata={'producer': 'Skia/PDF m156 Google Docs Renderer', 'cre

In [21]:
%pip install rank_bm25 faiss-cpu

Note: you may need to restart the kernel to use updated packages.


In [22]:
from langchain_community.retrievers import BM25Retriever  # keyword-based retriever
from langchain_community.vectorstores import FAISS  # vector database (FAISS)
from langchain_ollama import OllamaEmbeddings  # embedding model
from langchain_classic.retrievers import EnsembleRetriever  # merges two retrievers

# 1) BM25: build a keyword index straight from the chunk text (no embeddings needed)
bm25 = BM25Retriever.from_documents(chunks)  # your earlier chunks
bm25.k = 4  # return 4 chunks per query

# 2) FAISS: embed the chunks and build a vector index
embeddings = OllamaEmbeddings(model="nomic-embed-text")
faiss_store = FAISS.from_documents(chunks, embeddings)
faiss_retriever = faiss_store.as_retriever(search_kwargs={"k": 4})  # 4 chunks per query

# 3) Combine them; weights = importance of each (convention: they sum to 1)
ensemble = EnsembleRetriever(
    retrievers=[bm25, faiss_retriever],  # the two retrievers
    weights=[0.5, 0.5],  # equal importance
)

# 4) Test it
docs = ensemble.invoke("Who has experience with ERP development?")
for d in docs:
    print(d.metadata["candidate"], "|", d.page_content[:70].replace("\n", " "))

Ashikujjaman Shaikat Resume | CV of Ashikujjaman Shaikat Resume: - Backend Collaboration: Collaborat
Ashikujjaman Shaikat Resume | CV of Ashikujjaman Shaikat Resume: Md Ashikujjaman Shaikat PROFESSIONA
Ashikujjaman Shaikat Resume | CV of Ashikujjaman Shaikat Resume: - Created shared UI utilities to im
Ashikujjaman Shaikat Resume | CV of Ashikujjaman Shaikat Resume: - Integrated RESTful APIs with fron
Ashikujjaman Shaikat Resume | CV of Ashikujjaman Shaikat Resume: BJIT Limited ( Job Title: Senior So
Ashikujjaman Shaikat Resume | CV of Ashikujjaman Shaikat Resume: - Implemented dashboard visualizati
Ashikujjaman Shaikat Resume | CV of Ashikujjaman Shaikat Resume: - API Integration: Integrate RESTfu
